In [26]:
import numpy as np
import pandas as pd
import sklearn as sk
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels as sm

def fa(n,m, a,b):
    f,a = plt.subplots(n,m, figsize = (a,b))
    return f,a

In [27]:
prices = pd.concat([pd.read_csv(f"prices_round_4_day_{i}.csv", delimiter = ';') for i in [1, 2, 3]])
trades = pd.concat([pd.read_csv(f"trades_round_4_day_{i}.csv", delimiter = ';') for i in [1, 2, 3]])
prices = {product: prices[prices['product'] == product] for product in prices['product'].unique()}
hp, ve = prices['HYDROGEL_PACK'], prices['VELVETFRUIT_EXTRACT']

In [52]:
trades.columns

Index(['timestamp', 'buyer', 'seller', 'symbol', 'currency', 'price',
       'quantity'],
      dtype='str')

In [28]:
hp.head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
1,1,0,HYDROGEL_PACK,9950,13,9947.0,23.0,NaN,NaN,9966,13,9968.0,23.0,NaN,NaN,9958.0,0.0
20,1,100,HYDROGEL_PACK,9953,15,9950.0,21.0,NaN,NaN,9969,15,9971.0,21.0,NaN,NaN,9961.0,0.0
33,1,200,HYDROGEL_PACK,9953,14,9951.0,21.0,NaN,NaN,9969,14,9972.0,21.0,NaN,NaN,9961.0,0.0
47,1,300,HYDROGEL_PACK,9952,15,9950.0,26.0,NaN,NaN,9968,15,9971.0,26.0,NaN,NaN,9960.0,0.0
49,1,400,HYDROGEL_PACK,9953,10,9951.0,26.0,NaN,NaN,9969,10,9972.0,26.0,NaN,NaN,9961.0,0.0


In [46]:
import pandas as pd
import numpy as np

HORIZON = 200
MIN_TRADES = 30


def compute_informed_traders(price_df: pd.DataFrame,
                             trades_df: pd.DataFrame,
                             product: str,
                             fair_value_fn):
    
    # --- 1. Prep ---
    price_df = price_df.sort_values("timestamp").copy()
    trades_df = trades_df[trades_df["symbol"] == product].copy()
    
    # --- 2. Fair value ---
    price_df["FV"] = fair_value_fn(price_df)
    
    # forward FV (fixed horizon)
    price_df["FV_future"] = price_df["FV"].shift(-HORIZON)
    price_df["dFV"] = price_df["FV_future"] - price_df["FV"]

    # --- 🔥 FIXED forward rolling windows ---
    price_df["FV_max_future"] = (
        price_df["FV"]
        .shift(-HORIZON)
        .rolling(HORIZON, min_periods=1)
        .max()
    )

    price_df["FV_min_future"] = (
        price_df["FV"]
        .shift(-HORIZON)
        .rolling(HORIZON, min_periods=1)
        .min()
    )
    
    # --- 3. Merge (FIXED: include MFE/MAE inputs) ---
    price_small = price_df[
        ["timestamp", "FV", "FV_future", "dFV",
         "FV_max_future", "FV_min_future", "mid_price"]
    ]
    
    trades_df = trades_df.merge(price_small, on="timestamp", how="left")
    
    # --- FIXED: drop all required fields ---
    trades_df = trades_df.dropna(
        subset=["FV", "FV_future", "dFV", "FV_max_future", "FV_min_future"]
    )
    
    # --- 4. Expand trades ---
    buyer = trades_df.copy()
    buyer["trader"] = buyer["buyer"]
    buyer["sign"] = 1
    
    seller = trades_df.copy()
    seller["trader"] = seller["seller"]
    seller["sign"] = -1
    
    all_trades = pd.concat([buyer, seller], ignore_index=True)
    
    # --- 5. Core metrics ---
    all_trades["correct"] = (all_trades["sign"] * all_trades["dFV"]) > 0
    
    all_trades["edge"] = all_trades["sign"] * (
        all_trades["FV_future"] - all_trades["price"]
    )
    
    # --- decomposition ---
    all_trades["entry_edge"] = all_trades["sign"] * (
        all_trades["FV"] - all_trades["price"]
    )
    
    all_trades["timing_edge"] = all_trades["sign"] * (
        all_trades["FV_future"] - all_trades["FV"]
    )
    
    all_trades["signed_trade"] = all_trades["sign"]

    # --- MFE / MAE ---
    all_trades["mfe_edge"] = np.where(
        all_trades["sign"] == 1,
        all_trades["FV_max_future"] - all_trades["price"],
        all_trades["price"] - all_trades["FV_min_future"]
    )

    all_trades["mae_edge"] = np.where(
        all_trades["sign"] == 1,
        all_trades["FV_min_future"] - all_trades["price"],
        all_trades["price"] - all_trades["FV_max_future"]
    )

    all_trades["mfe_mae_ratio"] = all_trades["mfe_edge"] / (
        np.abs(all_trades["mae_edge"]) + 1e-6
    )
    
    # --- 6. Aggregation ---
    summary = all_trades.groupby("trader").agg(
        trades=("edge", "count"),
        hit_rate=("correct", "mean"),
        
        avg_edge=("edge", "mean"),
        edge_std=("edge", "std"),
        
        avg_entry_edge=("entry_edge", "mean"),
        entry_edge_sd=("entry_edge", "std"),
        
        avg_timing_edge=("timing_edge", "mean"),
        timing_edge_sd=("timing_edge", "std"),
        
        avg_mfe=("mfe_edge", "mean"),
        avg_mae=("mae_edge", "mean"),
        mfe_mae_ratio=("mfe_mae_ratio", "mean"),
    )
    
    # --- 7. Info ratios ---
    summary["info_ratio"] = summary["avg_edge"] / summary["edge_std"].replace(0, np.nan)
    summary["entry_info_ratio"] = summary["avg_entry_edge"] / summary["entry_edge_sd"].replace(0, np.nan)
    summary["timing_info_ratio"] = summary["avg_timing_edge"] / summary["timing_edge_sd"].replace(0, np.nan)
    
    # --- 8. Safe correlation ---
    def trader_corr(x):
        if len(x) < MIN_TRADES:
            return np.nan
        
        s = x["signed_trade"].to_numpy()
        d = x["dFV"].to_numpy()
        
        mask = ~np.isnan(s) & ~np.isnan(d)
        s, d = s[mask], d[mask]
        
        if len(s) < MIN_TRADES:
            return np.nan
        
        s_std = np.std(s)
        d_std = np.std(d)
        
        if s_std == 0 or d_std == 0:
            return np.nan
        
        cov = np.mean((s - s.mean()) * (d - d.mean()))
        return cov / (s_std * d_std)
    
    summary["corr_future"] = all_trades.groupby("trader").apply(trader_corr)
    
    # --- 9. Filter ---
    summary = summary[summary["trades"] >= MIN_TRADES]
    
    # --- 10. Rank ---
    summary = summary.sort_values(
        ["avg_timing_edge", "mfe_mae_ratio", "corr_future"],
        ascending=False
    )
    
    return summary, all_trades

In [47]:
def fair_value_velvet(df: pd.DataFrame) -> pd.Series:
    """
    Same as in r3-final-14.py but modified for use with a df.
    """

    # Extract columns
    bid_p3 = df["bid_price_3"]
    bid_v3 = df["bid_volume_3"]
    
    bid_p2 = df["bid_price_2"]
    bid_v2 = df["bid_volume_2"]
    
    ask_p3 = df["ask_price_3"]
    ask_v3 = df["ask_volume_3"]
    
    ask_p2 = df["ask_price_2"]
    ask_v2 = df["ask_volume_2"]

    # --- Valid masks (only include if BOTH price and volume exist) ---
    bid3_valid = bid_p3.notna() & bid_v3.notna()
    bid2_valid = bid_p2.notna() & bid_v2.notna()
    
    ask3_valid = ask_p3.notna() & ask_v3.notna()
    ask2_valid = ask_p2.notna() & ask_v2.notna()

    # --- Total value (only include valid entries) ---
    total = (
        (bid_p3 * bid_v3).where(bid3_valid, 0) +
        (bid_p2 * bid_v2).where(bid2_valid, 0) +
        (ask_p3 * ask_v3).where(ask3_valid, 0) +
        (ask_p2 * ask_v2).where(ask2_valid, 0)
    )

    # --- Total volume ---
    count = (
        bid_v3.where(bid3_valid, 0) +
        bid_v2.where(bid2_valid, 0) +
        ask_v3.where(ask3_valid, 0) +
        ask_v2.where(ask2_valid, 0)
    )

    fv = total / count

    fv[count == 0] = np.nan

    return fv

def fair_value_hydro(df: pd.DataFrame) -> pd.Series:
    """
    Exact equivalent of compute_fair_value but operating on a DataFrame.
    Preserves prev_makers state across rows (timestamps).
    """

    df = df.sort_values("timestamp").copy()
    
    fv = []
    prev_makers: Dict = {}

    for _, row in df.iterrows():
        
        # --- Extract levels (NaN-safe) ---
        def get(p, v, sign=1):
            if pd.notna(p) and pd.notna(v):
                return p, sign * v
            return None, None

        bid_price_1, bid_volume_1 = get(row["bid_price_1"], row["bid_volume_1"])
        bid_price_2, bid_volume_2 = get(row["bid_price_2"], row["bid_volume_2"])
        bid_price_3, bid_volume_3 = get(row["bid_price_3"], row["bid_volume_3"])

        ask_price_1, ask_volume_1 = get(row["ask_price_1"], row["ask_volume_1"], sign=-1)
        ask_price_2, ask_volume_2 = get(row["ask_price_2"], row["ask_volume_2"], sign=-1)
        ask_price_3, ask_volume_3 = get(row["ask_price_3"], row["ask_volume_3"], sign=-1)

        # --- initialise filtered makers ---
        f_mbp1 = f_mbv1 = f_mbp2 = f_mbv2 = None
        f_map1 = f_mav1 = f_map2 = f_mav2 = None

        # --- BID SIDE (exact logic) ---
        if bid_price_3 is not None:
            f_mbp1, f_mbv1 = bid_price_2, bid_volume_2
            f_mbp2, f_mbv2 = bid_price_3, bid_volume_3

        elif bid_price_2 is not None:
            if bid_volume_1 is not None and (bid_volume_1 < 10 or abs(bid_price_1 - bid_price_2) >= 5):
                if bid_volume_2 >= 20:
                    f_mbp2, f_mbv2 = bid_price_2, bid_volume_2
                else:
                    f_mbp1, f_mbv1 = bid_price_2, bid_volume_2
            else:
                f_mbp1, f_mbv1 = bid_price_1, bid_volume_1
                f_mbp2, f_mbv2 = bid_price_2, bid_volume_2

        elif bid_price_1 is not None:
            if bid_volume_1 >= 20:
                f_mbp2, f_mbv2 = bid_price_1, bid_volume_1
            elif bid_volume_1 >= 10:
                f_mbp1, f_mbv1 = bid_price_1, bid_volume_1

        # --- ASK SIDE (exact logic) ---
        if ask_price_3 is not None:
            f_map1, f_mav1 = ask_price_2, ask_volume_2
            f_map2, f_mav2 = ask_price_3, ask_volume_3

        elif ask_price_2 is not None:
            if ask_volume_1 is not None and (ask_volume_1 < 10 or abs(ask_price_1 - ask_price_2) >= 5):
                if ask_volume_2 >= 20:
                    f_map2, f_mav2 = ask_price_2, ask_volume_2
                else:
                    f_map1, f_mav1 = ask_price_2, ask_volume_2
            else:
                f_map1, f_mav1 = ask_price_1, ask_volume_1
                f_map2, f_mav2 = ask_price_2, ask_volume_2

        elif ask_price_1 is not None:
            if ask_volume_1 >= 20:
                f_map2, f_mav2 = ask_price_1, ask_volume_1
            elif ask_volume_1 >= 10:
                f_map1, f_mav1 = ask_price_1, ask_volume_1

        # --- Persist memory ---
        new_prev = dict(prev_makers)

        if f_mbp1 is not None:
            new_prev["maker_bid_price_1"]  = f_mbp1
            new_prev["maker_bid_volume_1"] = f_mbv1
        if f_mbp2 is not None:
            new_prev["maker_bid_price_2"]  = f_mbp2
            new_prev["maker_bid_volume_2"] = f_mbv2
        if f_map1 is not None:
            new_prev["maker_ask_price_1"]  = f_map1
            new_prev["maker_ask_volume_1"] = f_mav1
        if f_map2 is not None:
            new_prev["maker_ask_price_2"]  = f_map2
            new_prev["maker_ask_volume_2"] = f_mav2

        # --- Retrieve ---
        mbp1 = new_prev.get("maker_bid_price_1")
        mbp2 = new_prev.get("maker_bid_price_2")
        map1 = new_prev.get("maker_ask_price_1")
        map2 = new_prev.get("maker_ask_price_2")

        # --- Fair value ---
        if mbp1 is not None and mbp2 is not None and map1 is not None and map2 is not None:
            fair = (mbp1 + mbp2 + map1 + map2) / 4
        elif mbp1 is not None and map1 is not None:
            fair = (mbp1 + map1) / 2
        elif mbp2 is not None and map2 is not None:
            fair = (mbp2 + map2) / 2
        else:
            fair = np.nan

        fv.append(fair)
        prev_makers = new_prev  # update state

    return pd.Series(fv, index=df.index)

ve_summary, ve_trades = compute_informed_traders(
    ve, trades, "VELVETFRUIT_EXTRACT", fair_value_velvet
)

hp_summary, hp_trades = compute_informed_traders(
    hp, trades, "HYDROGEL_PACK", fair_value_hydro
)

In [50]:
ve_summary.sort_values(by='avg_timing_edge', ascending=False).head(10)

,trades,hit_rate,avg_edge,edge_std,avg_entry_edge,entry_edge_sd,avg_timing_edge,timing_edge_sd,avg_mfe,avg_mae,mfe_mae_ratio,info_ratio,entry_info_ratio,timing_info_ratio,corr_future
trader,,,,,,,,,,,,,,,
Mark 55,2438,0.506563,-2.068231,20.273899,-2.430734,19.797441,0.362503,20.234791,19.718895,-24.598296,1.933904,-0.102014,-0.122780,0.017915,0.017927
Mark 67,340,0.488235,1.602070,19.652390,1.314939,18.753431,0.287131,21.085793,22.311416,-22.304608,752944.902446,0.081520,0.070117,0.013617,NaN
Mark 14,1414,0.494342,2.150856,20.173077,2.021587,19.365202,0.129269,20.462568,24.870856,-20.096112,686002.059948,0.106620,0.104393,0.006317,0.007147
Mark 49,256,0.460938,-0.800668,18.705219,-0.487394,18.065383,-0.313274,20.849817,22.572526,-21.818685,2.120330,-0.042805,-0.026979,-0.015025,-0.003208
Mark 01,924,0.466450,2.331386,20.464433,3.223376,20.666560,-0.891989,19.913025,24.150529,-19.179397,720243.878394,0.113924,0.155971,-0.044794,-0.043986
Mark 22,264,0.518939,-1.867035,20.890296,-0.882956,19.201841,-0.984079,19.813434,23.110483,-21.762492,3.065015,-0.089373,-0.045983,-0.049667,-0.064519


In [49]:
hp_summary.head()

,trades,hit_rate,avg_edge,edge_std,avg_entry_edge,entry_edge_sd,avg_timing_edge,timing_edge_sd,avg_mfe,avg_mae,mfe_mae_ratio,info_ratio,entry_info_ratio,timing_info_ratio,corr_future
trader,,,,,,,,,,,,,,,
Mark 22,57,0.561404,0.421053,40.623411,-4.105263,40.455662,4.526316,36.849437,31.706140,-39.214912,3.070015,0.010365,-0.101476,0.122833,0.139478
Mark 38,3060,0.488562,-7.198856,39.890323,-7.311193,39.125679,0.112337,34.540234,29.264461,-44.207843,181623.241526,-0.180466,-0.186864,0.003252,0.003302
Mark 14,3003,0.506161,7.327506,39.872022,7.527889,39.074705,-0.200383,34.495415,44.445138,-29.075591,413345.852356,0.183776,0.192654,-0.005809,-0.005853


In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Callable, Tuple
from collections import defaultdict

# =========================
# CONFIG
# =========================
HORIZON = 50  # future timesteps to evaluate edge
MIN_TRADES = 30
OPTION_PREFIX = "VEV_"
UNDERLYING = "VELVETFRUIT_EXTRACT"

# =========================
# DATA PREP
# =========================
def clean_prices(prices: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    cleaned = {}

    for product, df in prices.items():
        df = df.copy().sort_values("timestamp")

        # Fill missing bid/ask via forward fill
        bid_cols = [c for c in df.columns if "bid_price" in c]
        ask_cols = [c for c in df.columns if "ask_price" in c]

        df[bid_cols] = df[bid_cols].ffill()
        df[ask_cols] = df[ask_cols].ffill()

        # Recompute mid if missing
        if "mid_price" not in df or df["mid_price"].isna().any():
            df["mid_price"] = (df["bid_price_1"] + df["ask_price_1"]) / 2

        # Compute returns
        df["future_mid"] = df["mid_price"].shift(-HORIZON)
        df["future_return"] = df["future_mid"] - df["mid_price"]

        cleaned[product] = df

    return cleaned


def clean_trades(trades: pd.DataFrame) -> pd.DataFrame:
    trades = trades.copy()
    trades = trades.sort_values("timestamp")
    trades.rename(columns={"symbol": "product"}, inplace=True)
    return trades


# =========================
# OPTION PARSING
# =========================
def parse_strike(product: str) -> float:
    if product.startswith(OPTION_PREFIX):
        return float(product.split("_")[1])
    return np.nan


def is_option(product: str) -> bool:
    return product.startswith(OPTION_PREFIX)


# =========================
# FAIR VALUE (SIMPLE)
# =========================
def fair_value_option(underlying_price: float, strike: float) -> float:
    """Simple intrinsic value approximation"""
    return max(0.0, underlying_price - strike)


# =========================
# MERGE TRADES WITH PRICES
# =========================
def merge_trades_prices(
    prices: Dict[str, pd.DataFrame],
    trades: pd.DataFrame
) -> pd.DataFrame:

    merged_rows = []

    for product, df in prices.items():
        tdf = trades[trades["product"] == product]

        if tdf.empty:
            continue

        merged = pd.merge_asof(
            tdf.sort_values("timestamp"),
            df.sort_values("timestamp"),
            on="timestamp",
            direction="backward",
            suffixes=("", "_price")
        )

        # 🔑 ensure consistent product column
        merged["product"] = product

        merged_rows.append(merged)

    return pd.concat(merged_rows, ignore_index=True)


# =========================
# INFORMED TRADER METRICS
# =========================
def compute_trader_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Measures:
    - directional accuracy
    - avg PnL vs future price
    - timing edge
    """

    trader_stats = defaultdict(lambda: {
        "pnl": 0.0,
        "correct": 0,
        "total": 0,
        "early_alpha": []
    })

    for _, row in df.iterrows():
        buyer = row["buyer"]
        seller = row["seller"]

        future_ret = row["future_return"]
        qty = row["quantity"]
        price = row["price"]

        # Buyer profits if price goes up
        buyer_pnl = future_ret * qty
        seller_pnl = -future_ret * qty

        for trader, pnl, direction in [
            (buyer, buyer_pnl, 1),
            (seller, seller_pnl, -1)
        ]:
            stats = trader_stats[trader]
            stats["pnl"] += pnl
            stats["total"] += 1

            if future_ret * direction > 0:
                stats["correct"] += 1

            stats["early_alpha"].append(pnl)

    # Build DataFrame
    rows = []
    for trader, stats in trader_stats.items():
        if stats["total"] < MIN_TRADES:
            continue

        rows.append({
            "trader": trader,
            "total_trades": stats["total"],
            "accuracy": stats["correct"] / stats["total"],
            "avg_pnl": stats["pnl"] / stats["total"],
            "total_pnl": stats["pnl"],
            "alpha_std": np.std(stats["early_alpha"])
        })

    return pd.DataFrame(rows).sort_values("total_pnl", ascending=False)


# =========================
# BEHAVIOUR CLASSIFICATION
# =========================
def classify_traders(metrics: pd.DataFrame) -> pd.DataFrame:
    """
    Label traders:
    - informed
    - noise
    - market maker
    """

    conditions = []

    for _, row in metrics.iterrows():
        if row["accuracy"] > 0.6 and row["avg_pnl"] > 0:
            label = "informed"
        elif row["accuracy"] < 0.45:
            label = "noise"
        else:
            label = "market_maker"

        conditions.append(label)

    metrics["type"] = conditions
    return metrics


# =========================
# SIGNAL EXTRACTION
# =========================
def build_signals(df: pd.DataFrame, informed_traders: set) -> pd.DataFrame:
    """
    Create signal:
    +1 if informed traders net buying
    -1 if selling
    """

    df = df.copy()

    df["signed_qty"] = 0

    df.loc[df["buyer"].isin(informed_traders), "signed_qty"] += df["quantity"]
    df.loc[df["seller"].isin(informed_traders), "signed_qty"] -= df["quantity"]

    signal = df.groupby(["timestamp", "product"])["signed_qty"].sum().reset_index()
    signal["signal"] = np.sign(signal["signed_qty"])

    return signal


# =========================
# BACKTEST SIGNAL
# =========================
def backtest_signal(signal_df: pd.DataFrame,
                    prices: Dict[str, pd.DataFrame]) -> pd.DataFrame:

    results = []

    for product, df in prices.items():
        sig = signal_df[signal_df["product"] == product]

        if sig.empty:
            continue

        merged = pd.merge_asof(
            sig.sort_values("timestamp"),
            df.sort_values("timestamp"),
            on="timestamp",
            direction="forward"
        )

        merged["future_mid"] = merged["mid_price"].shift(-HORIZON)
        merged["ret"] = merged["future_mid"] - merged["mid_price"]

        merged["strategy_pnl"] = merged["signal"] * merged["ret"]

        results.append({
            "product": product,
            "avg_pnl": merged["strategy_pnl"].mean(),
            "sharpe": merged["strategy_pnl"].mean() / (merged["strategy_pnl"].std() + 1e-6)
        })

    return pd.DataFrame(results)


# =========================
# MAIN PIPELINE
# =========================
def run_analysis(prices: Dict[str, pd.DataFrame],
                 trades: pd.DataFrame):

    prices_clean = clean_prices(prices)
    trades_clean = clean_trades(trades)

    merged = merge_trades_prices(prices_clean, trades_clean)

    metrics = compute_trader_metrics(merged)
    metrics = classify_traders(metrics)

    informed = set(metrics[metrics["type"] == "informed"]["trader"])

    signal = build_signals(merged, informed)
    results = backtest_signal(signal, prices_clean)

    return metrics, results


# =========================
# RUN
# =========================
metrics, results = run_analysis(prices, trades)

print("\n=== TRADER METRICS ===")
print(metrics.head(20))

print("\n=== STRATEGY RESULTS ===")
print(results)

KeyError: 'product'